In [1]:
# IMPORT LIBRARIES
import joblib
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report


# PARAMETERS
RANDOM_SEED = 13           # reproducibility
THRESHOLD = 0.5            # default prediction threshold (can tune)

# HYPERPARAMETERS (optimized for imbalanced dataset)
rf_params = {
    "n_estimators": 200,
    "max_depth": 10,
    "min_samples_split": 10,
    "min_samples_leaf": 5,
    "max_features": "sqrt",
    "class_weight": "balanced",
    "random_state": RANDOM_SEED,
    "n_jobs": -1
}

# LOAD DATA
baseline = pd.read_csv('baseline_features.csv')
dynamic = pd.read_csv('dynamic_features.csv')
data = pd.concat([baseline, dynamic], ignore_index=True)

# DROP rows where target is NaN
data = data.dropna(subset=['leak'])

# FEATURES & TARGET
X = data.drop(columns=['leak']).values
y = data['leak'].astype(int).values
feature_names = data.drop(columns=['leak']).columns

# TRAIN-TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_SEED, stratify=y
)

# QUICK CHECK
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train distribution:", np.bincount(y_train))
print("y_test distribution:", np.bincount(y_test))

# DEFINE AND TRAIN RANDOM FOREST
rf_model = RandomForestClassifier(**rf_params)
rf_model.fit(X_train, y_train)

# TRAINING AND TEST ACCURoACY
train_score = rf_model.score(X_train, y_train)
test_score = rf_model.score(X_test, y_test)
print(f"\nRandom Forest Training Accuracy: {train_score:.4f}")
print(f"Random Forest Test Accuracy: {test_score:.4f}")

# FEATURE IMPORTANCES
print("\nFeature Importances:")
importances = rf_model.feature_importances_
for name, importance in zip(feature_names, importances):
    print(f"{name}: {importance:.4f}")

# --- PREDICTIONS WITH THRESHOLD ---
y_proba = rf_model.predict_proba(X_test)[:, 1]        # probability of leak
y_pred = (y_proba >= THRESHOLD).astype(int)          # apply threshold

# --- EVALUATION METRICS ---
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

print("\nEvaluation Metrics:")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")

# CONFUSION MATRIX
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

# DETAILED CLASSIFICATION REPORT
print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))

# Save the trained model
joblib.dump(rf_model, "rf_model.pkl")
print("Random Forest model saved as rf_model.pkl")

X_train shape: (23788, 12)
X_test shape: (10196, 12)
y_train distribution: [23049   739]
y_test distribution: [9879  317]

Random Forest Training Accuracy: 0.8724
Random Forest Test Accuracy: 0.8736

Feature Importances:
time: 0.0091
node: 0.1765
rolling_mean: 0.0567
rolling_min: 0.0510
rolling_std: 0.0208
elevation: 0.1208
degree: 0.1156
pipe_length_avg: 0.0685
pipe_length_max: 0.0549
pipe_diameter_avg: 0.0788
pipe_diameter_max: 0.0410
pipe_age_avg: 0.2063

Evaluation Metrics:
Precision: 0.1920
Recall:    0.9558
F1-Score:  0.3198
ROC-AUC:   0.9418

Confusion Matrix:
[[8604 1275]
 [  14  303]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9984    0.8709    0.9303      9879
           1     0.1920    0.9558    0.3198       317

    accuracy                         0.8736     10196
   macro avg     0.5952    0.9134    0.6251     10196
weighted avg     0.9733    0.8736    0.9113     10196

Random Forest model saved as rf_model.pkl
